# Experiment 1: recomputing every table, question by question

This notebook is a **thin shell over `ex1_results.py`**. Every metric, file path
and check comes from that module by import, so a fix in one place fixes both and
the two can never disagree. Nothing is retyped.

Put the notebook **beside `ex1_results.py`**, in `fourarm/analysis/ex1/`.

## Order

The sections follow the seven questions of the results section, not the order
the tables happen to be defined in the script.

| Q | Question | Artefact |
|---|---|---|
| 1 | With everything stated, can the models apply it? | competence profile, Legal-Arm Control |
| 2 | What does removing the declared width cost? | `tab:ex1:design`, `tab:ex1:gaps` |
| 3 | Can the object's name substitute for the number? | same two tables |
| 4 | Does a false name mislead, with and without the width? | `tab:ex1:swap` |
| 5 | Does the rule text add anything once the number is there? | `tab:ex1:composition` |
| 6 | Does the model register the loss? | `tab:ex1:signatures`, `fig:ex1:convergence` |
| 7 | Which parts of the effect depend on the object set? | `tab:ex1:castb` |

## How to use it

- Run **Setup** once. Everything below depends on it.
- Every table cell runs with `write=False`, so nothing touches disk until the
  final write-out cell, which is gated behind a flag you must set yourself.
- Cells marked **CHECK** test a claim the prose makes. They are the reason the
  notebook exists rather than just running the script.

## Setup

In [1]:
import os, sys, json, math, collections, importlib, pathlib

# Find ex1_results.py: beside the notebook, or anywhere above it.
here = pathlib.Path.cwd()
for cand in [here] + list(here.parents):
    if (cand / "ex1_results.py").exists():
        sys.path.insert(0, str(cand))
        break
else:
    raise SystemExit("ex1_results.py not found. Put this notebook beside it.")

import ex1_results as R
importlib.reload(R)          # re-run this cell after editing the script

ROOT = R.find_root()
FILES, IMAGE, SCHEMES, MISSING = R.resolve_files(ROOT)
if MISSING:
    R.report_missing(ROOT, MISSING)
REF = R.reference_lines(ROOT)

print("root      %s" % ROOT)
print("run files %d (%s)" % (len(FILES), dict(SCHEMES)))
print("image-on  %s" % IMAGE)
print("cast A    width-blind %.1f, chance %.1f"
      % (REF["casta"]["width_blind"], REF["casta"]["chance"]))
print("cast B    width-blind %.1f, chance %.1f"
      % (REF["castb"]["width_blind"], REF["castb"]["chance"]))

root      /Users/erinsarlak/Downloads/MastersDissertation/fourarm
run files 30 ({'new': 30})
image-on  out/ex1_casta_gpt_nowidth-anon_r3_image.jsonl
cast A    width-blind 74.9, chance 30.5
cast B    width-blind 69.2, chance 34.6


In [2]:
# The integrity gate. Nothing below is trustworthy if this does not pass.
R.integrity_gate(ROOT, FILES, verbose=True)


Integrity gate
--------  ------  ------------------  --------------------------------------  ----  -------  --------  ------  ----
cast      model   condition           file                                    rows  repeats  balanced  errors  rung
--------  ------  ------------------  --------------------------------------  ----  -------  --------  ------  ----
casta     gemini  Anonymous           ex1_casta_gemini_anon_r3.jsonl          486   3        yes       0       ok  
casta     gemini  Full Information    ex1_casta_gemini_full_r3.jsonl          486   3        yes       0       ok  
casta     gemini  Legal-Arm Control   ex1_casta_gemini_givenset_r1.jsonl      162   1        yes       0       ok  
casta     gemini  No Rules            ex1_casta_gemini_norules_r3.jsonl       486   3        yes       0       ok  
casta     gemini  No Width            ex1_casta_gemini_nowidth_r3.jsonl       486   3        yes       0       ok  
casta     gemini  No Width + Anon.    ex1_casta_gemini_n

In [3]:
# Helpers. show() renders as a DataFrame when pandas is present and falls back
# to the script's plain printer when it is not.
try:
    import pandas as pd
    pd.set_option("display.max_colwidth", 60)
    HAVE_PANDAS = True
except ImportError:
    HAVE_PANDAS = False

def show(headers, rows, title=""):
    if HAVE_PANDAS:
        if title:
            print(title)
        return pd.DataFrame(rows, columns=headers)
    R.print_table(title, headers, rows)

def rows_for(model, cond, cast="casta"):
    # The raw run rows for one cell.
    return R.load(ROOT, FILES[(cast, model, cond)])

MODELS = R.MODELS
probes = json.load(open(os.path.join(ROOT, "probes/ex1_v2.json")))["probes"]

print("helpers ready. models:", MODELS)
print("conditions:", list(R.CONDITIONS))
print("cast A probes:", len(probes))

helpers ready. models: ['gemini', 'gpt', 'qwen']
conditions: ['full', 'anon', 'swap', 'nowidth', 'nowidth-anon', 'nowidth-swap', 'norules', 'givenset']
cast A probes: 162


---

# Q1. With everything stated, can the models apply it?

Two things have to come out of this question, and neither is in the current
chapter's baseline passage.

1. **The three-way split between models.** Nothing later makes sense without it.
   The width gap needs a baseline to be a gap from, and Qwen's flat response
   looks like a null rather than a floor until the baseline is on the page.
2. **That the failures differ in kind, not only in rate.** Gemini violates
   nothing, GPT violates grasp and route, Qwen violates four causes at once.

Legal-Arm Control belongs here rather than in a separate controls subsection,
because it is what licenses the reading of Qwen: hand Qwen the legal set and it
selects correctly, so its deficit is in deriving legality, not in format.

### 1a. The competence profile

In [4]:
headers = ["Model", "Picking trials correct", "Zero-legal refused",
           "Grasp-binding legality", "With legal set printed"]
rows, store = [], {}
for m in MODELS:
    full  = rows_for(m, "full")
    given = rows_for(m, "givenset")

    kp, np_ = R.legality_any_cause(ROOT, FILES, m)   # any binding cause
    kr, nr  = R.refusal_trial(full)
    kg, ng  = R.legality(full)                        # grasp-binding subset
    kl, nl  = R.legality(given)

    rows.append([R.MODEL_LABEL[m],
                 "%d/%d = %.1f" % (kp, np_, R.pct(kp, np_)),
                 "%d/%d = %.1f" % (kr, nr, R.pct(kr, nr)),
                 R.fmt_ci(kg, ng),
                 R.fmt_ci(kl, nl)])
    store[m] = dict(pick=(kp, np_), refuse=(kr, nr), grasp=(kg, ng),
                    given=(kl, nl))

show(headers, rows, "Competence profile at Full Information")


Competence profile at Full Information
------  ----------------------  ------------------  ----------------------  ----------------------
Model   Picking trials correct  Zero-legal refused  Grasp-binding legality  With legal set printed
------  ----------------------  ------------------  ----------------------  ----------------------
Gemini  378/378 = 100.0         101/108 = 93.5      100.0 [98.7, 100.0]     100.0 [96.2, 100.0]   
GPT     358/365 = 98.1          96/108 = 88.9       97.5 [95.0, 98.8]       100.0 [96.1, 100.0]   
Qwen    286/378 = 75.7          0/108 = 0.0         75.3 [70.1, 80.0]       95.8 [89.8, 98.4]     
------  ----------------------  ------------------  ----------------------  ----------------------


In [5]:
# Where each model's Full Information errors fall. This is the evidence for
# "the failures differ in kind", so it should be inspected, not assumed.
for m in MODELS:
    onzero, onpick = collections.Counter(), collections.Counter()
    for r in rows_for(m, "full"):
        vc = r.get("violation_cause")
        if not vc:
            continue
        (onzero if r.get("zero_legal") else onpick)[vc] += 1
    print("%-7s picking states    %s" % (R.MODEL_LABEL[m], dict(onpick) or "none"))
    print("%-7s zero-legal states %s" % ("", dict(onzero) or "none"))

Gemini  picking states    none
        zero-legal states {'no_route': 7}
GPT     picking states    {'no_route': 2, 'grasp': 5}
        zero-legal states {'no_route': 8, 'grasp': 2, 'delicate': 2}
Qwen    picking states    {'arm_state': 26, 'reach': 21, 'grasp': 14, 'delicate': 30, 'no_route': 1}
        zero-legal states {'grasp': 27, 'arm_state': 41, 'reach': 31, 'delicate': 9}


### 1b. Legal-Arm Control, and the 91/91 figure that does not reproduce

In [6]:
# The pooled "unlisted arm" share mixes two different failures. Split it.
IDLE = {}
for p in probes:
    pv = p["provenance"]
    IDLE[(pv["source"], pv["seq"], pv["round"])] = {
        a["name"] for a in p["state"]["arms"]
        if a.get("state") == "IDLE" and not a.get("disabled")}

headers = ["Model", "Picking proposals", "unlisted", "Zero-legal proposals",
           "unlisted"]
rows = []
for m in MODELS:
    c = collections.Counter()
    for r in rows_for(m, "givenset"):
        arm = (r.get("decision") or {}).get("arm")
        if not arm:
            continue
        where = "zero" if r.get("zero_legal") else "pick"
        c[(where, arm in IDLE[R.scene_key(r)])] += 1
    tp = c[("pick", True)]  + c[("pick", False)]
    tz = c[("zero", True)]  + c[("zero", False)]
    rows.append([R.MODEL_LABEL[m], tp, c[("pick", False)], tz, c[("zero", False)]])

show(headers, rows,
     "Legal-Arm Control: proposals naming an arm not on the printed list")


Legal-Arm Control: proposals naming an arm not on the printed list
------  -----------------  --------  --------------------  --------
Model   Picking proposals  unlisted  Zero-legal proposals  unlisted
------  -----------------  --------  --------------------  --------
Gemini  126                0         0                     0       
GPT     118                0         0                     0       
Qwen    126                5         34                    31      
------  -----------------  --------  --------------------  --------


In [7]:
# CHECK. The old chapter says Qwen "fails the Legal-Arm Control outright,
# 91 of 91". Recompute the pooled figure and confirm what it actually is.
for m in MODELS:
    tot = unl = 0
    for r in rows_for(m, "givenset"):
        arm = (r.get("decision") or {}).get("arm")
        if not arm:
            continue
        tot += 1
        unl += arm not in IDLE[R.scene_key(r)]
    print("  %-7s %d unlisted of %d proposals (%.1f%%)"
          % (R.MODEL_LABEL[m], unl, tot, R.pct(unl, tot)))
print("\nIf Qwen is not 91/91, the chapter sentence has to be rewritten.")
print("Legal-Arm Control is ONE repeat. Footnote that, or run Qwen at three.")

  Gemini  0 unlisted of 126 proposals (0.0%)
  GPT     0 unlisted of 118 proposals (0.0%)
  Qwen    36 unlisted of 160 proposals (22.5%)

If Qwen is not 91/91, the chapter sentence has to be rewritten.
Legal-Arm Control is ONE repeat. Footnote that, or run Qwen at three.


### 1c. The lift from printing the legal set

In [8]:
print("Legality with the legal set printed, minus Full Information\n")
for m in MODELS:
    d = R.newcombe(*store[m]["given"], *store[m]["grasp"])
    print("  %-7s %+.1f [%+.1f, %+.1f]%s"
          % (R.MODEL_LABEL[m], d[0], d[1], d[2],
             "" if R.spans_zero(d[1], d[2]) else "   EXCLUDES ZERO"))

Legality with the legal set printed, minus Full Information

  Gemini  +0.0 [-3.8, +1.3]
  GPT     +2.5 [-1.7, +5.0]
  Qwen    +20.5 [+12.9, +26.4]   EXCLUDES ZERO


### 1d. The constraint-difficulty null, as prose not a float

In [9]:
# The script still builds tab:ex1:constraints. Twelve cells, three of which are
# the delicacy row at n=9. Print it, then print the sentence that replaces it.
R.table_constraints(ROOT, FILES, show=True, write=False)


tab:ex1:constraints   Legality by constraint, Full Information
---------------  ----  ------------------  -------------  ------------  ------------
Constraint       Rule  Reasoning demanded  Gemini         GPT           Qwen        
---------------  ----  ------------------  -------------  ------------  ------------
Reach            R4    list lookup         100.0 (n=213)  99.0 (n=205)  77.9 (n=213)
Grasp            R3    numeric comparison  100.0 (n=153)  96.7 (n=151)  73.9 (n=153)
Delicacy         R3    boolean check       100.0 (n=9)    100.0 (n=6)   44.4 (n=9)  
All constraints  --    --                  100.0 (n=378)  98.1 (n=365)  75.7 (n=378)
---------------  ----  ------------------  -------------  ------------  ------------
reach minus grasp:
  Gemini  +0.0 [-1.8, +2.4]
  GPT     +2.3 [-0.8, +6.6]
  Qwen    +4.1 [-4.7, +13.1]
Delicacy binds only nine states. Draw no conclusion from that row.


In [10]:
print("Reach (a list lookup) against grasp (a numeric comparison), "
      "Full Information:\n")
for m in MODELS:
    k_r = n_r = k_g = n_g = 0
    for r in rows_for(m, "full"):
        if r.get("zero_legal") or r.get("result") not in ("valid", "rejected"):
            continue
        if r.get("binding_cause") == "reach":
            n_r += 1; k_r += r["result"] == "valid"
        elif r.get("binding_cause") == "grasp":
            n_g += 1; k_g += r["result"] == "valid"
    d = R.newcombe(k_r, n_r, k_g, n_g)
    print("  %-7s reach %5.1f (n=%d), grasp %5.1f (n=%d), "
          "difference %+.1f [%+.1f, %+.1f]"
          % (R.MODEL_LABEL[m], R.pct(k_r, n_r), n_r, R.pct(k_g, n_g), n_g,
             d[0], d[1], d[2]))
print("\nThe widest interval is the arithmetic penalty the data could NOT have")
print("detected. State that resolution: a null without one is not a result.")

Reach (a list lookup) against grasp (a numeric comparison), Full Information:

  Gemini  reach 100.0 (n=213), grasp 100.0 (n=153), difference +0.0 [-1.8, +2.4]
  GPT     reach  99.0 (n=205), grasp  96.7 (n=151), difference +2.3 [-0.8, +6.6]
  Qwen    reach  77.9 (n=213), grasp  73.9 (n=153), difference +4.1 [-4.7, +13.1]

The widest interval is the arithmetic penalty the data could NOT have
detected. State that resolution: a null without one is not a result.


---

# Q2. What does removing the declared width cost?

One 3x2 table serves Q2, Q3 and Q4: identity true, withheld or swapped, crossed
with width present or absent. Drawing two separate 2x2 grids would print the
Full Information and No Width cells twice.

In [11]:
R.table_design(ROOT, FILES, REF, show=True, write=False)


tab:ex1:design   Legality across the design, cast A
------  ------------------  -------  --------  -------------------  ---  -----  ------------
Model   Condition           Width    Identity  Legality             n    Scene  Neg. control
------  ------------------  -------  --------  -------------------  ---  -----  ------------
Gemini  Full Information    present  true      100.0 [98.7, 100.0]  288  100.0  100.0       
Gemini  Anonymous           present  withheld  100.0 [98.7, 100.0]  288  100.0  100.0       
Gemini  Swapped Names       present  swapped   100.0 [98.7, 100.0]  288  100.0  100.0       
Gemini  No Width            absent   true      72.2 [66.8, 77.1]    288  71.9   100.0       
Gemini  No Width + Anon.    absent   withheld  75.3 [70.1, 80.0]    288  77.1   100.0       
Gemini  No Width + Swapped  absent   swapped   76.0 [70.7, 80.5]    287  75.0   100.0       
Gemini  No Rules            --       --        100.0 [98.7, 100.0]  288  100.0  100.0       
Gemini  Legal-Arm

[['Gemini',
  'full',
  'Full Information',
  'present',
  'true',
  288,
  288,
  '100.00',
  '100.00',
  90,
  90,
  '100.00'],
 ['Gemini',
  'anon',
  'Anonymous',
  'present',
  'withheld',
  288,
  288,
  '100.00',
  '100.00',
  90,
  90,
  '100.00'],
 ['Gemini',
  'swap',
  'Swapped Names',
  'present',
  'swapped',
  288,
  288,
  '100.00',
  '100.00',
  90,
  90,
  '100.00'],
 ['Gemini',
  'nowidth',
  'No Width',
  'absent',
  'true',
  208,
  288,
  '72.22',
  '71.88',
  90,
  90,
  '100.00'],
 ['Gemini',
  'nowidth-anon',
  'No Width + Anon.',
  'absent',
  'withheld',
  217,
  288,
  '75.35',
  '77.08',
  90,
  90,
  '100.00'],
 ['Gemini',
  'nowidth-swap',
  'No Width + Swapped',
  'absent',
  'swapped',
  218,
  287,
  '75.96',
  '75.00',
  90,
  90,
  '100.00'],
 ['Gemini',
  'norules',
  'No Rules',
  '--',
  '--',
  288,
  288,
  '100.00',
  '100.00',
  90,
  90,
  '100.00'],
 ['Gemini',
  'givenset',
  'Legal-Arm Control',
  '--',
  '--',
  96,
  96,
  '100.00',
  '10

In [12]:
R.table_contrasts(ROOT, FILES, show=True, write=False)


tab:ex1:gaps   Contrasts, trial level (base minus manipulated)
----------------  ----------------  --------------------  --------------------  ------------------
Family            Level             Gemini                GPT                   Qwen              
----------------  ----------------  --------------------  --------------------  ------------------
Width removal     names true        +27.8 [+22.7, +33.2]  +24.0 [+18.5, +29.6]  -1.4 [-8.3, +5.6] 
Width removal     names anonymised  +24.7 [+19.8, +29.9]  +21.9 [+16.8, +27.2]  +4.2 [-2.9, +11.2]
Width removal     names swapped     +24.0 [+19.3, +29.3]  +19.9 [+14.7, +25.3]  +0.3 [-6.7, +7.4] 
Identity removed  width present     +0.0 [-1.3, +1.3]     -1.0 [-3.7, +1.5]     -1.4 [-8.3, +5.6] 
Identity removed  width absent      -3.1 [-10.3, +4.1]    -3.1 [-10.2, +4.0]    +4.2 [-2.9, +11.2]
Identity swapped  width present     +0.0 [-1.3, +1.3]     +0.1 [-2.8, +3.0]     +0.0 [-7.0, +7.0] 
Identity swapped  width absent      -3.7 [-10

In [13]:
# CHECK. Does each width-absent cell sit ON the width-blind line, or merely
# near it? The reference line is the strongest device in the chapter, so the
# claim should be tested rather than eyeballed.
wb = REF["casta"]["width_blind"]
print("width-blind line %.1f, chance floor %.1f\n"
      % (wb, REF["casta"]["chance"]))
for m in MODELS:
    for cond in ("nowidth", "nowidth-anon", "nowidth-swap"):
        k, n = R.legality(rows_for(m, cond))
        lo, hi = R.wilson(k, n)
        print("  %-7s %-14s %5.1f [%5.1f, %5.1f]  %s"
              % (R.MODEL_LABEL[m], cond, R.pct(k, n), lo, hi,
                 "includes the line" if lo <= wb <= hi else "EXCLUDES the line"))

width-blind line 74.9, chance floor 30.5

  Gemini  nowidth         72.2 [ 66.8,  77.1]  includes the line
  Gemini  nowidth-anon    75.3 [ 70.1,  80.0]  includes the line
  Gemini  nowidth-swap    76.0 [ 70.7,  80.5]  includes the line
  GPT     nowidth         73.6 [ 68.1,  78.4]  includes the line
  GPT     nowidth-anon    76.7 [ 71.4,  81.2]  includes the line
  GPT     nowidth-swap    77.6 [ 72.4,  82.1]  includes the line
  Qwen    nowidth         76.7 [ 71.5,  81.2]  includes the line
  Qwen    nowidth-anon    72.6 [ 67.1,  77.4]  includes the line
  Qwen    nowidth-swap    75.0 [ 69.7,  79.6]  includes the line


---

# Q3. Can the object's name substitute for the number?

Same two tables, read left to right instead of top to bottom. This question is
a null, so it needs its resolution stated alongside it.

In [14]:
R.report_interaction(ROOT, FILES, show=True)


Interaction, width gap by identity level
------  ---------------------  ------------------  ------------------
Model   against anonymisation  against swap        resolution +/- pts
------  ---------------------  ------------------  ------------------
Gemini  +3.1 [-4.1, +10.4]     +3.7 [-3.5, +11.0]  7                 
GPT     +2.1 [-5.5, +9.6]      +4.1 [-3.6, +11.7]  8                 
Qwen    -5.6 [-15.5, +4.4]     -1.7 [-11.6, +8.2]  10                
------  ---------------------  ------------------  ------------------
Every interval spans zero, so no interaction is DETECTABLE at this resolution.
That is not the same as no interaction existing.


In [15]:
# CHECK. All four identity contrasts, at both denominators, in one place.
# The claim is that none of them excludes zero at either.
print("identity contrasts, trial then scene\n")
for m in MODELS:
    for base, manip, label in (("full", "anon",         "removed, width present"),
                               ("nowidth", "nowidth-anon", "removed, width absent"),
                               ("full", "swap",         "swapped, width present"),
                               ("nowidth", "nowidth-swap", "swapped, width absent")):
        t = R.newcombe(*R.legality(rows_for(m, base)),
                       *R.legality(rows_for(m, manip)))
        s = R.newcombe(*R.scene_legality(rows_for(m, base)),
                       *R.scene_legality(rows_for(m, manip)))
        print("  %-7s %-24s trial %+5.1f [%+5.1f, %+5.1f]  scene %+5.1f [%+5.1f, %+5.1f]  %s"
              % (R.MODEL_LABEL[m], label, t[0], t[1], t[2], s[0], s[1], s[2],
                 "" if (R.spans_zero(t[1], t[2]) and R.spans_zero(s[1], s[2]))
                 else "<-- EXCLUDES ZERO SOMEWHERE"))

identity contrasts, trial then scene

  Gemini  removed, width present   trial  +0.0 [ -1.3,  +1.3]  scene  +0.0 [ -3.8,  +3.8]  
  Gemini  removed, width absent    trial  -3.1 [-10.3,  +4.1]  scene  -5.2 [-17.3,  +7.1]  
  Gemini  swapped, width present   trial  +0.0 [ -1.3,  +1.3]  scene  +0.0 [ -3.8,  +3.8]  
  Gemini  swapped, width absent    trial  -3.7 [-10.8,  +3.4]  scene  -3.1 [-15.4,  +9.3]  
  GPT     removed, width present   trial  -1.0 [ -3.7,  +1.5]  scene  -1.0 [ -6.3,  +3.9]  
  GPT     removed, width absent    trial  -3.1 [-10.2,  +4.0]  scene  -6.3 [-18.4,  +6.0]  
  GPT     swapped, width present   trial  +0.1 [ -2.8,  +3.0]  scene  +2.1 [ -3.7,  +8.4]  
  GPT     swapped, width absent    trial  -4.0 [-11.1,  +3.1]  scene  -7.4 [-19.4,  +4.9]  
  Qwen    removed, width present   trial  -1.4 [ -8.3,  +5.6]  scene  +1.0 [-10.9, +13.0]  
  Qwen    removed, width absent    trial  +4.2 [ -2.9, +11.2]  scene  +5.2 [ -7.0, +17.2]  
  Qwen    swapped, width present   trial  

---

# Q4. Does a false name mislead, with and without the width?

The directional test. Name-following requires the two swap directions to move in
**opposite** directions. The statistic is the separation between them, judged
against the drift on the untouched control objects printed beside it.

In [16]:
R.table_swap(ROOT, FILES, show=True, write=False)


tab:ex1:swap   Change in Franka share by swap direction
------  -------------  ---------------------  ---------------------  ------------------  -------------------
Model   Width          Wide obj, narrow name  Narrow obj, wide name  Untouched controls  Separation         
------  -------------  ---------------------  ---------------------  ------------------  -------------------
Gemini  width present  +0.0                   -5.3                   +0.6                +5.3 [-8.3, +19.0] 
Gemini  width absent   -5.9                   -6.8                   +0.6                +0.8 [-15.9, +17.5]
GPT     width present  +1.1                   -6.7                   -2.2                +7.8 [-7.4, +23.0] 
GPT     width absent   -3.4                   -1.6                   +0.3                -1.8 [-17.7, +14.0]
Qwen    width present  +2.3                   -3.0                   +4.7                +5.3 [-4.7, +15.4] 
Qwen    width absent   +1.0                   -6.2                   +7

In [17]:
# The plan promises the swap "reported on legality and refusal". The script
# computes legality but not refusal for the swap cells, so compute it here and
# either put it in the chapter or drop the promise.
headers = ["Model", "Full Info", "Swapped", "gap", "No Width", "No Width + Swapped", "gap"]
rows = []
for m in MODELS:
    a, b = R.refusal_trial(rows_for(m, "full")), R.refusal_trial(rows_for(m, "swap"))
    c, d = R.refusal_trial(rows_for(m, "nowidth")), R.refusal_trial(rows_for(m, "nowidth-swap"))
    rows.append([R.MODEL_LABEL[m],
                 "%.1f" % R.pct(*a), "%.1f" % R.pct(*b), R.fmt_gap(a, b),
                 "%.1f" % R.pct(*c), "%.1f" % R.pct(*d), R.fmt_gap(c, d)])
show(headers, rows, "Correct refusal, trial level, swap against its own base")


Correct refusal, trial level, swap against its own base
------  ---------  -------  ------------------  --------  ------------------  -------------------
Model   Full Info  Swapped  gap                 No Width  No Width + Swapped  gap                
------  ---------  -------  ------------------  --------  ------------------  -------------------
Gemini  93.5       95.4     -1.9 [-8.7, +4.8]   50.9      50.0                +0.9 [-12.2, +14.0]
GPT     88.9       91.7     -2.8 [-11.1, +5.4]  53.7      55.6                -1.9 [-14.9, +11.2]
Qwen    0.0        0.0      +0.0 [-3.4, +3.4]   0.0       0.0                 +0.0 [-3.4, +3.4]  
------  ---------  -------  ------------------  --------  ------------------  -------------------


In [18]:
# CHECK. The strongest single sentence available for Q4: a model that scored
# perfectly on grasp-binding trials while every swapped name was lying, and
# named the false object in its own reasons while doing so.
for m in MODELS:
    k, n = R.legality(rows_for(m, "swap"))
    print("  %-7s swapped-names grasp-binding legality %d/%d = %.1f"
          % (R.MODEL_LABEL[m], k, n, R.pct(k, n)))
print("\nRun analysis/ex1/ex1_check_swap.py for the partner-naming evidence.")

  Gemini  swapped-names grasp-binding legality 288/288 = 100.0
  GPT     swapped-names grasp-binding legality 268/275 = 97.5
  Qwen    swapped-names grasp-binding legality 217/288 = 75.3

Run analysis/ex1/ex1_check_swap.py for the partner-naming evidence.


---

# Q5. Does the rule text add anything once the number is there?

No Rules withholds **two** rules together, R3 (grasp and delicacy) and R4
(reach). It is one manipulation on the rules axis and it cannot be decomposed.

**The empty cell.** No Rules is not crossed with No Width. The plan's first
draft justified this by saying the width-removed row "already sits at the floor,
so there is no room to fall further". The cell below tests that and it is false:
the row sits on the width-blind line, not the chance floor. Use the reason the
`prompts.py` docstring gives instead, which is correct: a drop from No Width to
No Width + No Rules would confound instruction loss with the width loss already
measured above it.

In [19]:
R.table_composition(ROOT, FILES, show=True, write=False)


T3  Violations by cause
------  ----------------  -----  -----  --------  ---------  --------  -----
Model   Condition         grasp  reach  delicate  arm_state  no_route  Total
------  ----------------  -----  -----  --------  ---------  --------  -----
Gemini  Full Information  0      0      0         0          7         7    
Gemini  No Width          133    0      0         0          0         133  
Gemini  No Rules          1      0      0         0          0         1    
GPT     Full Information  7      0      2         0          10        19   
GPT     No Width          121    0      0         0          3         124  
GPT     No Rules          28     0      3         0          5         36   
Qwen    Full Information  41     52     39        67         1         200  
Qwen    No Width          43     56     34        55         1         189  
Qwen    No Rules          52     59     43        58         1         213  
------  ----------------  -----  -----  --------  -

In [20]:
# CHECK. How much room is there below the width-removed row?
wb, ch = REF["casta"]["width_blind"], REF["casta"]["chance"]
print("width-blind line %.1f, chance floor %.1f, room between them %.1f points\n"
      % (wb, ch, wb - ch))
for m in MODELS:
    k, n = R.legality(rows_for(m, "nowidth"))
    print("  %-7s No Width %5.1f, which is %.1f points ABOVE the chance floor"
          % (R.MODEL_LABEL[m], R.pct(k, n), R.pct(k, n) - ch))
print('\n"No room to fall further" is not supported. Do not write it.')

width-blind line 74.9, chance floor 30.5, room between them 44.4 points

  Gemini  No Width  72.2, which is 41.7 points ABOVE the chance floor
  GPT     No Width  73.6, which is 43.1 points ABOVE the chance floor
  Qwen    No Width  76.7, which is 46.2 points ABOVE the chance floor

"No room to fall further" is not supported. Do not write it.


In [21]:
# Violations split by state type. A violation on a zero-legal state is a failure
# to refuse; on a picking state it is a wrong choice. Pooling hides that.
for m in MODELS:
    for cond in ("full", "nowidth", "norules"):
        onzero, onpick = collections.Counter(), collections.Counter()
        for r in rows_for(m, cond):
            vc = r.get("violation_cause")
            if not vc:
                continue
            (onzero if r.get("zero_legal") else onpick)[vc] += 1
        print("%-7s %-9s picking %-46s zero-legal %s"
              % (R.MODEL_LABEL[m], cond, dict(onpick) or "none",
                 dict(onzero) or "none"))
    print()

Gemini  full      picking none                                           zero-legal {'no_route': 7}
Gemini  nowidth   picking {'grasp': 80}                                  zero-legal {'grasp': 53}
Gemini  norules   picking none                                           zero-legal {'grasp': 1}

GPT     full      picking {'no_route': 2, 'grasp': 5}                    zero-legal {'no_route': 8, 'grasp': 2, 'delicate': 2}
GPT     nowidth   picking {'grasp': 74}                                  zero-legal {'no_route': 3, 'grasp': 47}
GPT     norules   picking {'grasp': 13}                                  zero-legal {'no_route': 5, 'grasp': 15, 'delicate': 3}

Qwen    full      picking {'arm_state': 26, 'reach': 21, 'grasp': 14, 'delicate': 30, 'no_route': 1} zero-legal {'grasp': 27, 'arm_state': 41, 'reach': 31, 'delicate': 9}
Qwen    nowidth   picking {'arm_state': 20, 'reach': 18, 'grasp': 16, 'delicate': 26, 'no_route': 1} zero-legal {'grasp': 27, 'arm_state': 35, 'reach': 38, 'delicat

---

# Q6. Does the model register the loss?

The synthesis. Legality says the models fail without the width. It does not say
whether they **notice**. Three signatures answer that:

1. **Does it say so?** No grasp-error reason admits the width is missing.
2. **Does it act so?** Correct refusal *falls*. A model that noticed it had less
   information should decline more.
3. **What does it do instead?** Franka share rises to arm-indifference.

Two corrections the cells below enforce. The reasons bound must be **per model**,
because Qwen's denominator is a third of GPT's. And this is a **two-model
claim**: Qwen refuses 0.0% with the width and 0.0% without it, so "it declines
less" has no content for Qwen. Give Qwen one sentence saying why it is exempt.

In [22]:
R.table_signatures(ROOT, FILES, show=True, write=False)


tab:ex1:signatures   Three signatures of an unregistered loss
------  -------------------------  -----------  -----------------  --------------------
Model   Signature                  Width shown  Width absent       Change [95% CI]     
------  -------------------------  -----------  -----------------  --------------------
Gemini  Reasons admitting the gap  --           0/256              --                  
        Correct refusal            91.7         50.0               +41.7 [+21.1, +58.1]
        Franka share               28.1         48.0 [43.3, 52.7]  +20.0 [+13.4, +26.3]
GPT     Reasons admitting the gap  --           0/237              --                  
        Correct refusal            88.9         52.8               +36.1 [+15.3, +53.2]
        Franka share               26.0         45.7 [41.0, 50.6]  +19.7 [+13.1, +26.1]
Qwen    Reasons admitting the gap  --           0/85               --                  
        Correct refusal            0.0          0.0      

[['Gemini', 0, 256, '91.67', '50.00', '28.05', '48.03'],
 ['GPT', 0, 237, '88.89', '52.78', '25.99', '45.74'],
 ['Qwen', 0, 85, '0.00', '0.00', '20.99', '21.81']]

### 6a. Signature 1, per model rather than pooled

In [23]:
print("Bound on reasons admitting the missing width\n")
tot_adm = tot_ex = 0
for m in MODELS:
    adm = ex = 0
    for cond in ("nowidth", "nowidth-anon"):
        for r in rows_for(m, cond):
            if r.get("violation_cause") != "grasp":
                continue
            ex += 1
            adm += bool(R.MISSING_INFO.search(r.get("model_reason") or ""))
    tot_adm += adm; tot_ex += ex
    bound = 100 * 3 / ex if (adm == 0 and ex) else float("nan")
    print("  %-7s %d admit of %3d examined, one-sided 95%% upper bound %.1f%%"
          % (R.MODEL_LABEL[m], adm, ex, bound))
print("  %-7s %d admit of %3d examined, one-sided 95%% upper bound %.2f%%"
      % ("pooled", tot_adm, tot_ex, 100 * 3 / tot_ex))
print("\nQwen's bound is several times weaker than the pooled figure.")
print("Do not let the pooled number stand in for all three models.")

Bound on reasons admitting the missing width

  Gemini  0 admit of 256 examined, one-sided 95% upper bound 1.2%
  GPT     0 admit of 237 examined, one-sided 95% upper bound 1.3%
  Qwen    0 admit of  85 examined, one-sided 95% upper bound 3.5%
  pooled  0 admit of 578 examined, one-sided 95% upper bound 0.52%

Qwen's bound is several times weaker than the pooled figure.
Do not let the pooled number stand in for all three models.


In [24]:
# Inspect the reason strings, so "not one admits the gap" is checkable rather
# than taken on trust from a regular expression.
seen = 0
for r in rows_for("gpt", "nowidth"):
    if r.get("violation_cause") != "grasp":
        continue
    print("-", (r.get("model_reason") or "")[:140])
    seen += 1
    if seen >= 8:
        break

- Franka_s can directly deliver the clamp within sw while preserving ur_w for the wood block.
- Franka_s directly handles the clamp and preserves ur_w for the wood block.
- Franka_s can directly deliver the clamp while preserving ur_w for exclusive heavy tasks.
- Franka_s can deliver the clamp directly while preserving ur_w for exclusive tasks.
- franka_s can directly move the large clamp to the tools basket.
- Franka_s can deliver the clamp directly, preserving ur_w for uniquely reachable tasks.
- Franka_s can deliver the clamp directly while preserving ur_w for exclusive tasks.
- Franka_s can directly deliver the clamp to the tools destination.


### 6b. Signature 2, refusal, and why Qwen is exempt

In [25]:
# CHECK. tab:ex1:signatures reports refusal at the SCENE denominator and the
# 21 August notes quote the TRIAL figures. Both are correct and they differ.
# Print them side by side so the chapter cannot quote one under the other's
# label. Scene is authoritative.
headers = ["Model", "Trial: full", "Trial: nowidth", "Trial change",
           "Scene: full", "Scene: nowidth", "Scene change", "Note"]
rows = []
for m in MODELS:
    at, bt = R.refusal_trial(rows_for(m, "full")), R.refusal_trial(rows_for(m, "nowidth"))
    a, b = R.refusal_scene(rows_for(m, "full")), R.refusal_scene(rows_for(m, "nowidth"))
    note = "never refuses at either level, signature not applicable" \
           if (at[0] == 0 and bt[0] == 0) else ""
    rows.append([R.MODEL_LABEL[m],
                 "%.1f" % R.pct(*at), "%.1f" % R.pct(*bt), R.fmt_gap(at, bt),
                 "%.1f" % R.pct(*a), "%.1f" % R.pct(*b), R.fmt_gap(a, b), note])
show(headers, rows, "Correct refusal, Full Information minus No Width")


Correct refusal, Full Information minus No Width
------  -----------  --------------  --------------------  -----------  --------------  --------------------  -------------------------------------------------------
Model   Trial: full  Trial: nowidth  Trial change          Scene: full  Scene: nowidth  Scene change          Note                                                   
------  -----------  --------------  --------------------  -----------  --------------  --------------------  -------------------------------------------------------
Gemini  93.5         50.9            +42.6 [+31.4, +52.5]  91.7         50.0            +41.7 [+21.1, +58.1]                                                         
GPT     88.9         53.7            +35.2 [+23.5, +45.6]  88.9         52.8            +36.1 [+15.3, +53.2]                                                         
Qwen    0.0          0.0             +0.0 [-3.4, +3.4]     0.0          0.0             +0.0 [-9.6, +9.6]     never refu

### 6c. Signature 3, and the arm-indifference test

In [26]:
# CHECK. Two Franka and two UR, so 50.0 is pure indifference between arm types.
# The claim "as if every object fits" is measurable, not rhetorical.
print("Franka share with the width withheld, against 50.0\n")
for m in MODELS:
    k, n = R.franka_share(rows_for(m, "nowidth"))
    lo, hi = R.wilson(k, n)
    print("  %-7s %5.1f [%5.1f, %5.1f]  %s"
          % (R.MODEL_LABEL[m], R.pct(k, n), lo, hi,
             "INCLUDES 50.0, indistinguishable from indifference"
             if lo <= 50.0 <= hi else "excludes 50.0"))
print("\nAdd a 50 per cent reference line to the third panel of fig:ex1:convergence")
print("so the reader can see this rather than take it on trust.")

Franka share with the width withheld, against 50.0

  Gemini   48.0 [ 43.3,  52.7]  INCLUDES 50.0, indistinguishable from indifference
  GPT      45.7 [ 41.0,  50.6]  INCLUDES 50.0, indistinguishable from indifference
  Qwen     21.8 [ 18.4,  25.7]  excludes 50.0

Add a 50 per cent reference line to the third panel of fig:ex1:convergence
so the reader can see this rather than take it on trust.


In [27]:
R.figure_convergence(ROOT, FILES, REF, show=True, write=False, draw=False)


F1  convergence figure data: 54 rows (3 models x 6 conditions x 3 measures)


---

# Q7. Which parts of the effect depend on the object set?

Direction reproduces, magnitude does not. Reported as a generalisation check,
not a replication.

**One dependency to clear.** The chapter's "the difference is not an exposure
artefact" paragraph currently cites `tab:ex1:objects`, and the pipeline
deliberately drops the per-object table because the width-ordering reading was
withdrawn. The last cell recomputes the exposure counts so that paragraph can
carry its numbers in prose instead of dangling.

In [28]:
R.table_castb(ROOT, FILES, REF, show=True, write=False)


tab:ex1:castb   Cast B, GPT only, 108 states
----------------  -----------------  ---  -----  ---------------  -------------------
Condition         Legality           n    Scene  Independent run  Gap from Full Info 
----------------  -----------------  ---  -----  ---------------  -------------------
Full Information  98.8 [95.8, 99.7]  169  100.0  98.1             --                 
No Width          90.1 [84.6, 93.8]  162  87.5   88.5             +8.7 [+3.9, +14.3] 
No Width + Anon.  88.5 [82.7, 92.5]  165  86.0   92.2             +10.3 [+5.3, +16.2]
----------------  -----------------  ---  -----  ---------------  -------------------
The independent run is a separate single-repeat execution of the same three cells at the same prompt version, not a subset of the three-repeat run. Two executions agree to within about a point on legality while disagreeing on roughly a fifth of individual states.
width-removal gap, GPT
  cast A  +24.0 [+18.5, +29.6]   width-blind line 74.9
  cast B  

In [29]:
# The independent single-repeat run against repeat 1 of the three-repeat run:
# same states, same prompt version, different session.
for cond in R.CASTB_CONDITIONS:
    a = {R.scene_key(r): r for r in R.load(ROOT, FILES[("castb_r1", "gpt", cond)])}
    b = {R.scene_key(r): r for r in R.load(ROOT, FILES[("castb", "gpt", cond)])
         if r.get("repeat", 1) == 1}
    diff = sum(1 for k in a
               if (a[k].get("result"), (a[k].get("decision") or {}).get("arm"))
               != (b[k].get("result"), (b[k].get("decision") or {}).get("arm")))
    print("  %-14s %3d of %3d states answered differently (%.0f%%)"
          % (cond, diff, len(a), 100 * diff / len(a)))

  full            25 of 108 states answered differently (23%)
  nowidth         19 of 108 states answered differently (18%)
  nowidth-anon    21 of 108 states answered differently (19%)


In [30]:
# Exposure counts for cast B, replacing the dropped tab:ex1:objects.
# How many times did each object come up as a candidate task at all?
setb = json.load(open(os.path.join(ROOT, "probes/ex1_setb_v1.json")))["probes"]
opportunities = collections.Counter()
for p in setb:
    for t in p["state"]["tasks"]:
        opportunities[t["object"]] += 1

true_object_b = {}
for p in setb:
    pv = p["provenance"]
    for t in p["state"]["tasks"]:
        true_object_b[(pv["source"], pv["seq"], pv["round"], t["id"])] = t["object"]

chosen, errors = collections.Counter(), collections.Counter()
for r in R.load(ROOT, FILES[("castb", "gpt", "nowidth")]):
    d = r.get("decision") or {}
    tid = d.get("task_id")
    if tid is None:
        continue
    pv = r["provenance"]
    o = true_object_b.get((pv["source"], pv["seq"], pv["round"], tid))
    if o is None:
        continue
    chosen[o] += 1
    errors[o] += r.get("result") == "rejected"

headers = ["Object", "States offering it", "Times chosen (No Width)", "Errors"]
rows = [[o.replace("ycb_", ""), opportunities[o], chosen[o], errors[o]]
        for o in sorted(opportunities, key=lambda k: -opportunities[k])]
show(headers, rows, "Cast B exposure, GPT, No Width")


Cast B exposure, GPT, No Width
-------------  ------------------  -----------------------  ------
Object         States offering it  Times chosen (No Width)  Errors
-------------  ------------------  -----------------------  ------
caster         101                 23                       1     
sugar_box      83                  43                       16    
t_connector    82                  51                       0     
scissors       69                  43                       3     
tuna_can       65                  10                       0     
bracket_small  61                  47                       0     
bleach         60                  5                        3     
screw_99       57                  16                       0     
foam_brick     52                  17                       1     
mac_n_cheese   37                  14                       0     
-------------  ------------------  -----------------------  ------


---

# Robustness checks

Not research questions. The image-on cell closes a limitation previously stated
in the chapter, so it earns a short paragraph rather than a table.

In [31]:
R.prose_image(ROOT, FILES, IMAGE, show=True)


Image on, GPT
  file                    ex1_casta_gpt_nowidth-anon_r3_image.jsonl
  rung in the data        L1-nowidth
  grasp-binding legality  75.4 [70.0, 80.0]  n=284
  image minus text-only   -1.3 [-8.3, +5.7]  spans zero
  CHECK THE PAIRING. This file is the nowidth-anon condition, which withholds
  names, so the matched text-only cell must be nowidth-anon and not nowidth.


In [32]:
R.reconcile(ROOT, FILES, show=True)
R.summarise_checks()


Reconciliation against the project status
------  -------------------------------  --------------------  --------------------  -----
Model   Contrast                         computed here         status doc            agree
------  -------------------------------  --------------------  --------------------  -----
Gemini  width gap, names true            +27.8 [+22.7, +33.2]  +27.8 [+22.7, +33.2]  yes  
GPT     width gap, names true            +24.0 [+18.5, +29.6]  +24.0 [+18.5, +29.6]  yes  
Qwen    width gap, names true            -1.4 [-8.3, +5.6]     -1.4 [-8.3, +5.6]     yes  
Gemini  width gap, anonymised            +24.7 [+19.8, +29.9]  +24.7 [+19.8, +29.9]  yes  
GPT     width gap, anonymised            +21.9 [+16.8, +27.2]  +21.9 [+16.8, +27.2]  yes  
Qwen    width gap, anonymised            +4.2 [-2.9, +11.2]    +4.2 [-2.9, +11.2]    yes  
Gemini  identity removed, width present  +0.0 [-1.3, +1.3]     +0.0 [-1.3, +1.3]     yes  
GPT     identity removed, width present  -1.0 [

0

---

# Writing out

Everything above ran with `write=False`, so nothing has touched disk. The files
land under the **package root**, not next to this notebook, so
`\input{tables/ex1_design}` keeps working from `main.tex`.

**The step that is easy to forget.** `main.tex` lives in Overleaf, not in the
repository, so writing the tables here does not update the thesis. Copy
`tables/` and `figures/` across after every regeneration, or the chapter will
compile stale numbers without complaining.

In [33]:
CONFIRM = False          # set True to write

if CONFIRM:
    R.WRITTEN.clear()
    R.table_constraints(ROOT, FILES, False, True)
    R.table_design(ROOT, FILES, REF, False, True)
    R.table_contrasts(ROOT, FILES, False, True)
    R.table_signatures(ROOT, FILES, False, True)
    R.table_swap(ROOT, FILES, False, True)
    R.table_composition(ROOT, FILES, False, True)
    R.table_castb(ROOT, FILES, REF, False, True)
    R.figure_convergence(ROOT, FILES, REF, False, True, True)
    for p in R.WRITTEN:
        print("wrote", p)
    print("\nIn main.tex:")
    for p in R.WRITTEN:
        if p.endswith(".tex"):
            print("    \\input{%s}" % p[:-len(".tex")])
    print("\nNow copy tables/ and figures/ into the Overleaf project.")
else:
    print("CONFIRM is False, nothing written. "
          "Set it True and re-run this cell when you are ready.")

CONFIRM is False, nothing written. Set it True and re-run this cell when you are ready.


---

## Decisions this notebook leaves open

1. **Q1.** Replace `tab:ex1:constraints` with the competence profile, keep both,
   or keep only the profile? Recommendation is replace, with the
   constraint-difficulty null surviving as the sentence printed in 1d.
2. **Q1.** Footnote that Legal-Arm Control is one repeat, or spend 324 calls
   running Qwen at three. Its cell carries the Qwen argument.
3. **Q4.** Does swap refusal go in the chapter, or does the plan drop its promise
   to report it?
4. **Q6.** Does `fig:ex1:convergence` lead and the table support, or the reverse?
5. **Q6.** Add the 50% indifference line to the Franka panel.
6. The one errored row in `ex1_casta_gemini_nowidth-swap_r3.jsonl`, 1 of 486.
   Footnote it or repair it.